# DefectForge M18 — SegFormer-B0 downstream segmentation

Thin Colab wrapper around `src/training/train_segmenter.py`. Run one object per notebook copy. The ninth `all_mixed` group is an alias of `filtered_syn` and is deliberately not rerun.

## 1. Runtime and GPU preflight

Choose **Runtime → Change runtime type → T4 GPU**. The cell stops before installation when CUDA is absent or total VRAM is below 14 GiB.

In [ ]:
import subprocess

import torch

assert torch.cuda.is_available(), 'Select a T4 GPU runtime first'
props = torch.cuda.get_device_properties(0)
total_gib = props.total_memory / 2**30
print(props.name, f'{total_gib:.1f} GiB')
assert total_gib >= 14, 'M18 requires a T4-class GPU with at least 14 GiB VRAM'
subprocess.run(['nvidia-smi'], check=True)

## 2. Mount Drive and stage one object locally

Before running, place `defectforge_m18_source.zip` and either `m18_seg_pcb1.zip` or `m18_seg_capsules.zip` in `MyDrive/sdg-portfolio/01-defectforge-visa/`. Set `OBJECT_NAME` below. Both archives are copied and extracted under `/content`; training never reads images from mounted Drive.

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

OBJECT_NAME = 'pcb1'  # change the second notebook copy to 'capsules'
assert OBJECT_NAME in {'pcb1', 'capsules'}
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/sdg-portfolio/01-defectforge-visa')
PROJECT_ROOT = Path('/content/defectforge')
DATA_ROOT = Path('/content/data/01-defectforge')
SOURCE_ARCHIVE = DRIVE_ROOT / 'defectforge_m18_source.zip'
DATA_ARCHIVE = DRIVE_ROOT / f'm18_seg_{OBJECT_NAME}.zip'
for archive in (SOURCE_ARCHIVE, DATA_ARCHIVE):
    assert archive.is_file(), f'Missing {archive.name} in Drive'
shutil.copy2(SOURCE_ARCHIVE, '/content/defectforge_m18_source.zip')
shutil.copy2(DATA_ARCHIVE, f'/content/m18_seg_{OBJECT_NAME}.zip')
subprocess.run(['unzip', '-q', '/content/defectforge_m18_source.zip', '-d', '/content'], check=True)
subprocess.run(['unzip', '-q', f'/content/m18_seg_{OBJECT_NAME}.zip', '-d', '/content/data'], check=True)
assert (PROJECT_ROOT / 'pyproject.toml').is_file()
assert (DATA_ROOT / 'raw/VisA' / OBJECT_NAME).is_dir()
assert (DATA_ROOT / 'm18_colab_selection.json').is_file()

## 3. Reproducible environment and packaged selection

No Colab Secret is required: the pinned NVIDIA SegFormer checkpoint is public. The cell installs the frozen environment, redirects `data_root` to `/content`, and injects the checksummed per-group packaged sample IDs. It does not change any training hyperparameter.

In [ ]:
import json

import yaml

subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=PROJECT_ROOT, check=True)
paths = yaml.safe_load((PROJECT_ROOT / 'configs/paths.yaml').read_text(encoding='utf-8'))
paths['data_root'] = str(DATA_ROOT)
paths['dotenv'] = '/content/defectforge-no-secrets.env'
COLAB_PATHS = PROJECT_ROOT / 'configs/paths_colab.yaml'
COLAB_PATHS.write_text(yaml.safe_dump(paths, sort_keys=False), encoding='utf-8')
selection = json.loads((DATA_ROOT / 'm18_colab_selection.json').read_text(encoding='utf-8'))
assert selection['object'] == OBJECT_NAME
segmenter = yaml.safe_load((PROJECT_ROOT / 'configs/segmenter.yaml').read_text(encoding='utf-8'))
for group_name, sample_ids in selection['groups'].items():
    synthetic = segmenter['groups'][group_name]['synthetic']
    synthetic['view'] = 'm18_colab_pool'
    synthetic.pop('inputs', None)
    synthetic['sample_ids_by_object'] = {OBJECT_NAME: sample_ids}
COLAB_CONFIG = PROJECT_ROOT / 'configs/segmenter_colab.yaml'
COLAB_CONFIG.write_text(yaml.safe_dump(segmenter, sort_keys=False), encoding='utf-8')

## 4. Dry-run, eight formal groups, and automatic resume

Each group has a unique Drive checkpoint directory. Reopening the same notebook resumes the latest checkpoint. The fixed budget is 500 optimizer steps per group. The current planning estimate is 20–40 minutes per group on T4, or roughly 3–6 hours for one object; record the actual wall time and the Colab compute-unit change shown in your account.

In [ ]:
import time

GROUPS = ('real_only', 'std_aug', 'unfiltered_syn', 'filtered_syn', 'full_real', 'procedural_only', 'copypaste_only', 'diffusion_only')
LOCAL_RUNS = DATA_ROOT / 'runs/seg'
DRIVE_RUNS = DRIVE_ROOT / 'runs/seg' / OBJECT_NAME
timings = {}
for group_name in GROUPS:
    run_name = f'm18_{group_name}_{OBJECT_NAME}_seed42'
    local_output = LOCAL_RUNS / run_name
    drive_output = DRIVE_RUNS / run_name
    drive_output.mkdir(parents=True, exist_ok=True)
    checkpoints = sorted(drive_output.glob('checkpoint-*'))
    if checkpoints and not local_output.exists():
        shutil.copytree(drive_output, local_output)
    subprocess.run(['uv', 'run', '--frozen', 'python', 'src/training/train_segmenter.py', '--paths', str(COLAB_PATHS), '--config', str(COLAB_CONFIG), '--object', OBJECT_NAME, '--group', group_name, '--mode', 'final', '--run-name', run_name, '--output-dir', str(local_output), '--dry-run'], cwd=PROJECT_ROOT, check=True)
    command = ['uv', 'run', '--frozen', 'python', 'src/training/train_segmenter.py', '--paths', str(COLAB_PATHS), '--config', str(COLAB_CONFIG), '--object', OBJECT_NAME, '--group', group_name, '--mode', 'final', '--run-name', run_name, '--output-dir', str(local_output), '--drive-sync', str(drive_output)]
    if checkpoints and not (local_output / 'training_report.json').is_file():
        command += ['--resume-from-checkpoint', 'latest']
    started = time.perf_counter()
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    timings[group_name] = time.perf_counter() - started
    print(group_name, f'{timings[group_name]:.1f}s')

## 5. Independently validate and collect one result ZIP

The validator rebuilds signatures, verifies the frozen test inventory and blocklist, checks every group and final SafeTensors model, confirms zero real defect pixels in `procedural_only`, and confirms that `all_mixed` was not rerun. Download `m18_seg_results_<object>.zip` after this cell finishes.

In [ ]:
LOCAL_RESULT = Path(f'/content/m18_seg_results_{OBJECT_NAME}')
LOCAL_RESULT.mkdir(parents=True, exist_ok=True)
validation = LOCAL_RESULT / f'm18_{OBJECT_NAME}_validation.json'
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/validate_segmenter_runs.py', '--paths', str(COLAB_PATHS), '--config', str(COLAB_CONFIG), '--run-root', str(LOCAL_RUNS), '--object', OBJECT_NAME, '--reload', '--output', str(validation)], cwd=PROJECT_ROOT, check=True)
(LOCAL_RESULT / 'timings.json').write_text(json.dumps(timings, indent=2, sort_keys=True), encoding='utf-8')
csv_path = PROJECT_ROOT / 'results/segmentation.csv'
assert csv_path.is_file()
shutil.copy2(csv_path, LOCAL_RESULT / f'segmentation_{OBJECT_NAME}.csv')
for group_name in GROUPS:
    run_name = f'm18_{group_name}_{OBJECT_NAME}_seed42'
    source = LOCAL_RUNS / run_name
    target = LOCAL_RESULT / 'runs' / run_name
    target.mkdir(parents=True, exist_ok=True)
    for filename in ('training_report.json', 'run_config.json', 'data_manifest.json'):
        shutil.copy2(source / filename, target / filename)
    shutil.copytree(source / 'final', target / 'final', dirs_exist_ok=True)
archive_base = Path(f'/content/m18_seg_results_{OBJECT_NAME}')
archive = Path(shutil.make_archive(str(archive_base), 'zip', LOCAL_RESULT))
DRIVE_RESULTS = DRIVE_ROOT / 'results/segmentation'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, DRIVE_RESULTS / archive.name)
print('Validated result archive:', DRIVE_RESULTS / archive.name)